# Train an SO101-Nexus policy with PPO on MuJoCo Warp

PPO on the GPU-parallel **MuJoCo Warp** backend. Pick a task in step 3.

Reference results on one RTX 5090: `WarpPickLift-v1` reaches 0.905 min, 0.952 mean, and 0.993 max final success across seeds 1, 2, and 3 at 30M steps; touch, look-at, and move solve in about a minute. Colab A100/L4 are slower; a free T4 works but is slow (lower `--num-envs`).

## 1. Check the GPU

Runtime -> Change runtime type -> GPU.

In [ ]:
!nvidia-smi -L

## 2. Install

The example scripts live in `examples/`, not the wheel, so clone the repo and install the `warp` + `train` extras.

If you hit `ModuleNotFoundError: No module named 'so101_nexus._reproducibility'` (or any `so101_nexus` submodule) in a later cell, it is not a stale wheel: `pip install -e` adds the package to `sys.path` via a `.pth` file that Python only reads at interpreter startup. This kernel was already running before the install cell executed, so `!python ...` cells (fresh subprocesses) pick up the new install but in-kernel `import so101_nexus...` cells do not. The install cell below also inserts `src/` onto `sys.path` directly so both paths work without a runtime restart.

In [ ]:
!git clone --depth 1 https://github.com/johnsutor/so101-nexus.git
%cd so101-nexus
!pip uninstall -y so101-nexus 2>/dev/null || true
!pip install -q -e ".[warp,train]" "imageio[ffmpeg]"
# `pip install -e` registers the package via a .pth file that Python only
# reads when an interpreter starts. This kernel was already running before
# the install above, so `!python ...` cells (fresh subprocesses) see the
# package but in-kernel `import so101_nexus...` cells below will not unless
# `src/` is also put on sys.path explicitly, here.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "src"))

## 3. Choose the environment

PickLift needs the most steps; the others solve fast.

In [ ]:
# Options: WarpPickLift-v1, WarpTouch-v1, WarpLookAt-v1, WarpMove-v1
ENV_ID = "WarpPickLift-v1"
STEPS = {"WarpPickLift-v1": 30_000_000}.get(ENV_ID, 5_000_000)
print(f"Training {ENV_ID} for {STEPS:,} env steps")

## 4. Launch TensorBoard

Run before training so the dashboard picks up the new run.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

## 5. Train

1024 GPU-parallel worlds, fixed-horizon episodes, entropy annealed 0.005 -> 0, staggered resets, obs/reward normalization, and a CleanRL-style optimizer budget. The entropy and LR schedules run over `--total-timesteps`, so the step budget is part of the recipe. `best_agent.pt` is saved on every success-rate improvement. Reduce `--num-envs` on smaller GPUs.

MuJoCo eval rendering is off on Colab (`--eval-freq 0`); step 6 evaluates on the Warp backend instead.

In [ ]:
!python examples/ppo_warp.py --exp-name colab --env-id {ENV_ID} \
    --total-timesteps {STEPS} --eval-freq 0

## 6. Evaluate the trained policy

Deterministic eval (no exploration noise) across 512 Warp worlds. Reports `success_rate(ever)`, hold rate at the final step, and mean return.

In [ ]:
CKPT = f"runs/{ENV_ID}__colab__*/best_agent.pt"
!python examples/eval_warp.py --env-id {ENV_ID} --checkpoint "{CKPT}"

## 7. Watch a sample rollout

The Warp backend runs 1024 worlds in parallel and does not render, so you cannot watch a single agent from training. This renders one deterministic episode of the trained policy in the matching MuJoCo backend (same policy weights and observation normalization, with a slight physics gap versus Warp). The mp4 plays inline below.


In [ ]:
# Render one deterministic rollout of the trained policy and play it inline.
import glob

from IPython.display import Video

from examples.ppo_warp import rollout_video_from_checkpoint

checkpoint = sorted(glob.glob(CKPT))[-1]
print(f"Rendering sample rollout from {checkpoint}")

metrics, video_path = rollout_video_from_checkpoint(
    checkpoint,
    ENV_ID,
    control_mode="pd_joint_delta_pos",
    episode_length=512,
    hidden_dim=256,
    seed=12345,
)
print(
    f"rollout success_rate={metrics['eval/success_rate']:.2f} "
    f"return={metrics['eval/return']:.2f} "
    f"ep_len={metrics['eval/ep_len']:.0f}"
)
Video(video_path)

## Next steps

- PickLift lift discovery is still stochastic on GPU; the default recipe was validated on seeds 1, 2, and 3, but trying another `--seed` is still a useful sanity check.
- Tuning: `examples/README.md` and the [Training with PPO](https://so101-nexus.com/docs/training/ppo) guide.